Intro to project

In [1]:
import joblib
from tqdm import tqdm
import pandas as pd
import numpy as np
from itertools import product

from sklearn.preprocessing import MinMaxScaler
from skorch import NeuralNetClassifier
from skorch.callbacks import EarlyStopping
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score

from sklearn.ensemble import RandomForestClassifier
import xgboost as xg
from lightgbm import LGBMClassifier
import torch
import torch.nn as nn
from scipy.special import logit, expit

from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt
from matplotlib.patches import Arc
from sklearn.calibration import calibration_curve

RANDOM_SEED = 694973
np.random.seed(RANDOM_SEED)

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

# for reproducibility
print("pandas: "+pd.__version__)
print("numpy: "+np.__version__)
import sklearn
print("sklearn: "+sklearn.__version__)
import xgboost
print("xgboost: "+xgboost.__version__)
import lightgbm
print("lgbm: "+lightgbm.__version__)
print("torch: "+torch.__version__)

pandas: 3.0.1
numpy: 2.4.3
sklearn: 1.9.0
xgboost: 3.2.0
lgbm: 4.6.0
torch: 2.5.1


In [2]:
features = ['distance_mm', 'angle_mm']
df=pd.read_csv('data/shot_probs_data.csv')
df = df.reset_index(drop=False)
scaler = joblib.load('data/scaler_da.pkl')

X_train = df.loc[df['is_train']==1][['distance_mm','angle_mm']].values
X_test = df.loc[df['is_train']!=1][['distance_mm','angle_mm']].values
y_train=df.loc[df['is_train']==1]['is_made'].values
y_test=df.loc[df['is_train']!=1]['is_made'].values

dunks = df.loc[(df['is_train']==1)&(df['shot_type']=='DUNK')]
print(np.mean(dunks['is_made']))

jumpers = df.loc[(df['is_train']==1)&(df['shot_type']=='JUMPER')]
print(np.mean(jumpers['is_made']))

hooks = df.loc[(df['is_train']==1)&(df['shot_type']=='HOOK')]
print(np.mean(hooks['is_made']))

layups = df.loc[(df['is_train']==1)&(df['shot_type']=='LAYUP')]
print(np.mean(layups['is_made']))

0.8855009483565821
0.3908853789305898
0.4855232781839169
0.5642752867570385


In [3]:
def compute_scores(target, preds):
    loss = log_loss(target, preds)
    auc = roc_auc_score(target, preds)
    binary_preds = (preds >= 0.5).astype(int)
    accuracy = accuracy_score(target, binary_preds)
    return loss, auc, accuracy

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

In [4]:
!nvidia-smi

Thu Jun  4 21:07:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        Off |   00000000:01:00.0  On |                  N/A |
|  0%   35C    P8             29W /  350W |    2129MiB /  24576MiB |     62%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [19]:
# xgboost params
xg_params = {
    'n_estimators':[100, 250, 500, 1000],
    'max_depth': [2, 3],
    'learning_rate': [.05, 0.1, 0.3],
    'reg_lambda':[1, 3],
    'reg_alpha': [0, 0.5],
}

fixed_params = {
    'subsample': 0.75,
    'device':'cuda',
    'eval_metric':'logloss',
    'n_jobs':1,
    'random_state':RANDOM_SEED,
}

xgb_model = xg.XGBClassifier(**fixed_params)
gs_xgb = GridSearchCV(
    estimator=xgb_model,
    param_grid=xg_params,
    scoring='f1_weighted',
    cv=cv_strat,
    n_jobs=-1,
    verbose=1
)

In [20]:
gs_xgb.fit(X_train, y_train)
best_xg = gs_xgb.best_estimator_
best_xg_params = gs_xgb.best_params_
print(best_xg_params)

Fitting 5 folds for each of 96 candidates, totalling 480 fits
{'learning_rate': 0.3, 'max_depth': 3, 'n_estimators': 100, 'reg_alpha': 0, 'reg_lambda': 1}


In [21]:
# make pred
def make_preds(model, X):
    return model.predict_proba(X)[:,1]

def score_model(model, X, y):
    preds = make_preds(model, X)
    loss, auc, acc = compute_scores(y, preds)
    print(f"Log Loss: {loss:.4f}, AUC: {auc:.4f}, Accuracy: {acc:.4f}")
    return None

score_model(best_xg, X_train, y_train)
score_model(best_xg, X_test, y_test)

Log Loss: 0.6503, AUC: 0.6456, Accuracy: 0.6239
Log Loss: 0.6512, AUC: 0.6430, Accuracy: 0.6236


In [29]:
# lgbm params
lgbm_params = {
    'n_estimators':[100, 250, 500, 1000],
    'max_depth': [2, 3, 4],
    'learning_rate': [0.01, 0.1, 0.3],
    'num_leaves': [7, 15, 31],
    'reg_lambda':[1, 5],
    'reg_alpha': [0, 0.5],
}

fixed_params = {
    'subsample': 0.75,
    'device':'gpu',
    'n_jobs':1,
    'random_state':RANDOM_SEED,
}

lgb_model = LGBMClassifier(**fixed_params)
gs_lgb = GridSearchCV(
    estimator=lgb_model,
    param_grid=lgbm_params,
    scoring='f1_weighted',
    cv=cv_strat,
    n_jobs=-1,
    verbose=0
)

In [ ]:
gs_lgb.fit(X_train, y_train)
best_lg = gs_lgb.best_estimator_
best_lg_params = gs_lgb.best_params_

In [33]:
# rf params
rf_params = {
    'n_estimators': [100, 200, 500],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 10, 50],
    'min_samples_leaf': [1, 5, 20],
    'max_features':[1, 2]
}
rf_model = RandomForestClassifier(n_jobs=1, random_state=RANDOM_SEED)
gs_rf = GridSearchCV(
    estimator=rf_model,
    param_grid=rf_params,
    scoring='f1_weighted',
    cv=cv_strat,
    n_jobs=-1
)

In [34]:
gs_rf.fit(X_train, y_train)
best_rf = gs_rf.best_estimator_
best_rf_params = gs_rf.best_params_

/home/jr/anaconda3/envs/pie/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [ ]:
score_model(best_rf, X_train, y_train)
score_model(best_rf, X_test, y_test) 

Log Loss: 0.6332, AUC: 0.6911, Accuracy: 0.6404
Log Loss: 0.6592, AUC: 0.6298, Accuracy: 0.6143


In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_sizes, dropout):
        super().__init__()
        layers = []
        inputs = 2 # two for angle and distance
        for h in hidden_sizes:
            layers += [
                nn.Linear(inputs, h), # connected layers
                nn.BatchNorm1d(h), # norm weights
                nn.ReLU(), # activation func
                nn.Dropout(dropout) # rando dropout neurons for generlization
            ]
            inputs = h
        layers.append(nn.Linear(inputs, 1)) # output layer
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

early_stop = EarlyStopping(
    monitor='valid_loss',
    patience=10,
    threshold=.001,
    lower_is_better=True
)

param_grid = {
    'module__hidden_sizes': [(16, 8), (32, 16), (64, 32, 16)],
    'module__dropout': [0.2, 0.4],
    'optimizer__lr': [.001, .01, 0.03],
    'batch_size': [512, 2048],
}

net = NeuralNetClassifier(
    module=MLP,
    criterion=nn.BCEWithLogitsLoss,
    optimizer=torch.optim.Adam,
    max_epochs=200,
    callbacks=[early_stop],
    device='cuda',
)

gs_nn = GridSearchCV(
    estimator=net,
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=cv_strat,
    n_jobs=1
)

In [ ]:
gs_nn.fit(
    X_train.astype('float32'),
    y_train.astype('int32')
)